# Crypto Portfolio Analytics

Discovers assets held in configured Hyperliquid wallets, refreshes public CoinGecko daily USD prices and market data, calculates correlations and constrained long/short full-Kelly weights, and reports portfolio performance, risk, and factor estimates. Implementation lives in `crypto_portfolio_analytics_implementation.ipynb`.


## Load implementation

Set the wallet addresses to analyze, then load the companion implementation and create the analytics object.


In [13]:
import json
from pathlib import Path

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]


implementation_path=Path('crypto_portfolio_analytics_implementation.ipynb')
implementation=json.loads(implementation_path.read_text(encoding='utf-8'))
namespace={'__name__':'crypto_portfolio_analytics'}
for cell in implementation['cells']:
    if cell['cell_type']=='code':
        source=''.join(cell['source'])
        if source.strip():
            exec(compile(source,f'{implementation_path} cell','exec'),namespace)
CryptoPortfolioAnalytics=namespace['CryptoPortfolioAnalytics']
analysis=CryptoPortfolioAnalytics(data_dir='data', wallet_addresses=WALLET_ADDRESSES)


## Market data refresh

Discovers nonzero wallet positions and spot balances, then refreshes saved prices and market-cap/volume history from CoinGecko without requiring an API key. Edit `WALLET_ADDRESSES` above to select wallets.


In [14]:
analysis.refresh_market_data()


CoinGecko rate limit (HTTP 429); retrying attempt 2/8 in 60.0 seconds.
CoinGecko rate limit (HTTP 429); retrying attempt 2/8 in 60.0 seconds.
CoinGecko rate limit (HTTP 429); retrying attempt 2/8 in 60.0 seconds.
CoinGecko rate limit (HTTP 429); retrying attempt 2/8 in 60.0 seconds.
Refreshed saved prices and market data: IMX 2 days, ONDO 2 days, XTEN 0 days, USDC 2 days, ZRO 2 days, BLUR 2 days, RSR 2 days, DYM 2 days, MERL 2 days, EIGEN 2 days, PENGU 2 days, AIXBT 2 days, MELANIA 2 days, VVV 2 days, KAITO 2 days, PROVE 2 days, HEMI 2 days, HYPE 2 days, BTC 2 days. Assets discovered from 2 wallet(s): IMX, ONDO, USDC, ZRO, BLUR, RSR, DYM, MERL, EIGEN, PENGU, AIXBT, MELANIA, VVV, KAITO, PROVE, HEMI, HYPE. Public historical requests are limited to the latest 365 days; older saved rows were retained.
Skipping wallet asset(s) without a unique CoinGecko ID: OP (ambiguous CoinGecko symbol), XPL (ambiguous CoinGecko symbol), BANANA (ambiguous CoinGecko symbol), ADA (ambiguous CoinGecko symbol

## Price history

Loads assets discovered from wallet positions and spot balances, plus BTC for benchmark comparisons.


In [15]:
analysis.load_price_history()


## Daily returns and correlations

Calculates the complete daily-return sample and Pearson correlations.


In [16]:
analysis.calculate_returns()


## Correlation heatmap

Displays the daily return correlation matrix as an interactive Plotly heatmap.


In [17]:
analysis.plot_correlations()


## Constrained long/short full-Kelly portfolio

Optimizes signed asset weights from historical returns; the remaining net balance is held as cash. Long/short signs are chosen by the optimizer and are not forced to match current wallet position sides.


In [18]:
analysis.optimize_portfolio()


Constrained full-Kelly weights, sorted by coin weight (%)


,Weight (%)
ZRO,15.00
HYPE,15.00
DYM,15.00
VVV,15.00
RSR,14.14
ONDO,3.86
IMX,2.00
BLUR,2.00
USDC,2.00
EIGEN,2.00


## Portfolio performance and risk metrics

Displays the Plotly portfolio-vs-BTC chart and performance and risk tables.


In [19]:
analysis.analyze_performance()


,Return
1D,3.60%
7D,-3.92%
M (1 month),-9.90%
Q (3 months),49.66%
Since Inception,221.73%


,Daily return
Average daily return,0.55%
Largest daily gain,59.87%
Largest daily loss,-22.10%


,All-Time High ($),Minimum Portfolio Value ($),Max Drawdown ($),Max Drawdown (%)
Since inception,$386.19,$100.00,-$99.27,-32.40%


,Sharpe,Beta,Alpha,Expected Return,Hist Vol (daily),Hist Vol (annualized),1D Hist VaR 95%,VaR 95% ($ / $100),1D Hist VaR 99%,VaR 99% ($ / $100),Risk-Free Rate,Max Drawdown
Constrained full-Kelly,1.697,0.808,205.67%,202.37%,6.09%,116.37%,6.40%,$6.40,16.61%,$16.61,5.00%,-32.40%


## CoinGecko factor model

Fits factor exposures and displays the annual factor estimates and model fit.


In [20]:
analysis.fit_factor_model()


Factor model: R²=82.1% | adjusted R²=81.7% | annual alpha=7.73% | n=283 days
Single-factor BTC beta=0.808 | factor-model BTC beta=3.036


,Beta,HAC t-stat,Factor premium (%/yr),Portfolio contribution (%/yr)
BTC market excess,3.036,11.35,-12.16%,-36.93%
Alt market - BTC,2.428,7.39,30.93%,75.12%
Small - large,0.551,7.37,13.74%,7.57%
Momentum,-0.014,-0.25,-178.21%,2.47%
High - low turnover,-0.634,-8.23,-173.77%,110.16%
Alpha,N/A,0.17,N/A,7.73%


,Factor model fit
Observations,283.000000
R-squared,0.820641
Adjusted R-squared,0.817403
Annual alpha,0.077251
BTC beta (factor model),3.036164
BTC beta (single factor),0.807765


## Save analysis outputs

Writes calculated tables and series to CSV files in `data/`.


In [21]:
analysis.save_outputs()


Saved portfolio, factor, market, and return outputs to data/.
